# 02c · Descriptive analysis per instance

Describes the prepared conversation dataset **before** the conversation graphs are
built: language distribution, conversation size, user participation, instance-level
activity, cross-instance visibility and toxicity.

An instance-specific conversation is identified by `(instance_toot_found_id, conversation_id)`,
since the same federated conversation can be observed on several instances.

Input: `<dataset>_data_with_sent_and_tox.csv` from notebook 02.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

## Configuration

In [ ]:
DATASET = "pleroma"  # or "mastodon"
DATA_DIR = Path("../../data") / DATASET
DATA_FILE = DATA_DIR / f"{DATASET}_data_with_sent_and_tox.csv"

FIGURES_DIR = Path("../../figures") / DATASET / "descriptive"
TABLES_DIR = FIGURES_DIR / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

TOXICITY_THRESHOLD = 0.5
CONVERSATION_KEY = ["instance_toot_found_id", "conversation_id"]

## Load and validate

In [ ]:
df = pd.read_csv(DATA_FILE, low_memory=False)

required_columns = {
    "created_at", "author_id", "instance_toot_belong_id", "conversation_id",
    "toot_id", "instance_toot_found_id", "root_toot", "lang", "toxicity",
}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

df["toxicity"] = pd.to_numeric(df["toxicity"], errors="coerce")

print(f"Toot observations:            {len(df):,}")
print(f"Unique toots:                 {df['toot_id'].nunique():,}")
print(f"Unique conversations:         {df['conversation_id'].nunique():,}")
print(f"Instance-conversation pairs:  {df[CONVERSATION_KEY].drop_duplicates().shape[0]:,}")
print(f"Origin instances:             {df['instance_toot_belong_id'].nunique():,}")
print(f"Observation instances:        {df['instance_toot_found_id'].nunique():,}")

## Plot helpers

In [ ]:
def describe_counts(values, title):
    summary = values.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99])
    print(title)
    print(summary.to_string())
    return summary


def plot_histogram(values, title, xlabel, bins=50, log_x=False, log_y=True, save_name=None):
    values = pd.Series(values).dropna()
    values = values[values >= 0]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(values, bins=bins)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Frequency")
    if log_x and (values > 0).any():
        ax.set_xscale("log")
    if log_y:
        ax.set_yscale("log")
    fig.tight_layout()
    if save_name:
        fig.savefig(FIGURES_DIR / save_name, dpi=300, bbox_inches="tight")
    plt.show()


def plot_cdf_ccdf(values, title_prefix, xlabel, save_prefix=None):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    values = values[np.isfinite(values) & (values >= 0)]
    if values.size == 0:
        print("No values available.")
        return

    x, counts = np.unique(np.sort(values), return_counts=True)
    n = counts.sum()
    cumulative = np.cumsum(counts)

    for kind, y, ylabel in [
        ("cdf", cumulative / n, "CDF: P(X ≤ x)"),
        ("ccdf", (n - (cumulative - counts)) / n, "CCDF: P(X ≥ x)"),
    ]:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.step(x, y, where="post")
        if np.all(x > 0):
            ax.set_xscale("log")
        if kind == "ccdf":
            ax.set_yscale("log")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title_prefix} — {kind.upper()}")
        fig.tight_layout()
        if save_prefix:
            fig.savefig(FIGURES_DIR / f"{save_prefix}_{kind}.png", dpi=300, bbox_inches="tight")
        plt.show()

## Language distribution

In [ ]:
language_counts = df["lang"].fillna("NA").astype(str).value_counts()
language_summary = pd.DataFrame({
    "count": language_counts,
    "percentage": language_counts / language_counts.sum() * 100,
})
display(language_summary)

fig, ax = plt.subplots(figsize=(9, 5))
language_summary.sort_values("percentage")["percentage"].plot(kind="barh", ax=ax)
ax.set_xlabel("Percentage of toot observations")
ax.set_ylabel("Language")
ax.set_title("Language distribution")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "language_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

## Authors per conversation

In [ ]:
authors_per_conversation = (
    df.dropna(subset=CONVERSATION_KEY + ["author_id"])
    .groupby(CONVERSATION_KEY)["author_id"]
    .nunique()
    .rename("n_authors")
    .reset_index()
)
describe_counts(authors_per_conversation["n_authors"], "Distinct authors per conversation")
plot_histogram(
    authors_per_conversation["n_authors"],
    "Authors per conversation", "Distinct authors",
    log_x=True, save_name="authors_per_conversation_histogram.png",
)
plot_cdf_ccdf(
    authors_per_conversation["n_authors"],
    "Authors per conversation", "Distinct authors per conversation",
    save_prefix="authors_per_conversation",
)

## Toots per conversation

In [ ]:
toots_per_conversation = (
    df.dropna(subset=CONVERSATION_KEY + ["toot_id"])
    .groupby(CONVERSATION_KEY)["toot_id"]
    .nunique()
    .rename("n_toots")
    .reset_index()
)
describe_counts(toots_per_conversation["n_toots"], "Distinct toots per conversation")
plot_histogram(
    toots_per_conversation["n_toots"],
    "Toots per conversation", "Distinct toots",
    log_x=True, save_name="toots_per_conversation_histogram.png",
)
plot_cdf_ccdf(
    toots_per_conversation["n_toots"],
    "Toots per conversation", "Distinct toots per conversation",
    save_prefix="toots_per_conversation",
)

## Conversations per instance

The number of conversation nodes available to each instance before edges are built.

In [ ]:
conversations_per_instance = (
    df.dropna(subset=CONVERSATION_KEY)
    .groupby("instance_toot_found_id")["conversation_id"]
    .nunique()
    .rename("n_conversations")
    .reset_index()
)
describe_counts(conversations_per_instance["n_conversations"], "Distinct conversations per instance")
display(conversations_per_instance.sort_values("n_conversations", ascending=False).head(10))
plot_histogram(
    conversations_per_instance["n_conversations"],
    "Conversations per instance", "Distinct conversations",
    log_x=True, save_name="conversations_per_instance_histogram.png",
)
plot_cdf_ccdf(
    conversations_per_instance["n_conversations"],
    "Conversations per instance", "Distinct conversations per instance",
    save_prefix="conversations_per_instance",
)

## Toots per author

A toot can be observed on several instances, so `(author_id, toot_id)` pairs are
de-duplicated first.

In [ ]:
toots_per_author = (
    df.dropna(subset=["author_id", "toot_id"])[["author_id", "toot_id"]]
    .drop_duplicates()
    .groupby("author_id")["toot_id"]
    .nunique()
    .rename("n_toots")
    .reset_index()
)
describe_counts(toots_per_author["n_toots"], "Distinct toots per author")
plot_histogram(
    toots_per_author["n_toots"],
    "Toots per author", "Distinct toots",
    log_x=True, save_name="toots_per_author_histogram.png",
)
plot_cdf_ccdf(toots_per_author["n_toots"], "Toots per author", "Distinct toots per author", save_prefix="toots_per_author")

## Cross-instance visibility

A toot is cross-instance when the instance it originates from differs from the
instance where it was observed (`instance_toot_belong_id != instance_toot_found_id`).

In [ ]:
toot_visibility = (
    df[["toot_id", "instance_toot_belong_id", "instance_toot_found_id"]]
    .dropna()
    .astype(str)
    .drop_duplicates()
)
cross_instance_toots = toot_visibility[
    toot_visibility["instance_toot_belong_id"] != toot_visibility["instance_toot_found_id"]
]
print(f"Toots observed on at least one foreign instance: {cross_instance_toots['toot_id'].nunique():,}")

federation_flows = (
    cross_instance_toots.groupby(["instance_toot_belong_id", "instance_toot_found_id"])["toot_id"]
    .nunique()
    .rename("n_toots")
    .reset_index()
    .sort_values("n_toots", ascending=False)
)
foreign_instances_per_toot = (
    cross_instance_toots.groupby("toot_id")["instance_toot_found_id"]
    .nunique()
    .rename("n_foreign_instances")
    .reset_index()
)

display(federation_flows.head(20))
describe_counts(foreign_instances_per_toot["n_foreign_instances"], "Foreign observation instances per toot")
plot_histogram(
    foreign_instances_per_toot["n_foreign_instances"],
    "Foreign-instance visibility per toot", "Number of foreign instances",
    log_x=True, save_name="foreign_instances_per_toot_histogram.png",
)

## Toxicity per instance

A toot is toxic when its Detoxify score is at least `TOXICITY_THRESHOLD`. Each toot is
counted once per observation instance.

In [ ]:
toxicity_base = (
    df[["instance_toot_found_id", "toot_id", "toxicity"]]
    .dropna()
    .drop_duplicates(subset=["instance_toot_found_id", "toot_id"])
    .copy()
)
toxicity_base["is_toxic"] = toxicity_base["toxicity"] >= TOXICITY_THRESHOLD

toxicity_per_instance = (
    toxicity_base.groupby("instance_toot_found_id")
    .agg(n_toots=("toot_id", "nunique"), n_toxic=("is_toxic", "sum"), toxic_rate=("is_toxic", "mean"))
    .reset_index()
)
describe_counts(toxicity_per_instance["n_toxic"], "Toxic toots per instance")
describe_counts(toxicity_per_instance["toxic_rate"], "Toxic-toot rate per instance")

MIN_INSTANCE_TOOTS = 20
display(toxicity_per_instance.sort_values("n_toxic", ascending=False).head(10))
display(
    toxicity_per_instance[toxicity_per_instance["n_toots"] >= MIN_INSTANCE_TOOTS]
    .sort_values("toxic_rate", ascending=False)
    .head(10)
)

In [ ]:
count_buckets = pd.cut(
    toxicity_per_instance["n_toxic"],
    bins=[0, 1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, np.inf],
    labels=["0", "1", "2–5", "6–10", "11–20", "21–50", "51–100", "101–200", "201–500", "501–1000", "1000+"],
    include_lowest=True,
).value_counts().sort_index()

rate_buckets = pd.cut(
    toxicity_per_instance["toxic_rate"],
    bins=[0, 0.01, 0.05, 0.10, 0.20, 0.50, 1.00],
    labels=["0–1%", "1–5%", "5–10%", "10–20%", "20–50%", "50–100%"],
    include_lowest=True,
).value_counts().sort_index()

for buckets, xlabel, title, name in [
    (count_buckets, "Toxic toots per instance", f"Toxic-toot counts (threshold = {TOXICITY_THRESHOLD})", "toxic_toot_count_buckets.png"),
    (rate_buckets, "Toxic-toot rate", "Toxic-toot rates", "toxic_rate_buckets.png"),
]:
    fig, ax = plt.subplots(figsize=(9, 5))
    buckets.plot(kind="bar", ax=ax)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of instances")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / name, dpi=300, bbox_inches="tight")
    plt.show()

## Toxic and non-toxic toots in the largest instances

In [ ]:
TOP_N = 30

composition = (
    toxicity_base.groupby(["instance_toot_found_id", "is_toxic"])["toot_id"]
    .nunique()
    .unstack(fill_value=0)
    .rename(columns={False: "non_toxic", True: "toxic"})
)
for column in ["non_toxic", "toxic"]:
    if column not in composition:
        composition[column] = 0
composition["total"] = composition["non_toxic"] + composition["toxic"]

fig, ax = plt.subplots(figsize=(12, 6))
composition.sort_values("total", ascending=False).head(TOP_N)[["non_toxic", "toxic"]].plot(
    kind="bar", stacked=True, ax=ax
)
ax.set_xlabel("Observation instance")
ax.set_ylabel("Unique toots")
ax.set_title(f"Toxic and non-toxic toots in the {TOP_N} largest instances")
ax.legend(title=None)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "toxic_non_toxic_largest_instances.png", dpi=300, bbox_inches="tight")
plt.show()

## Export tables

In [ ]:
for name, table in {
    "authors_per_conversation": authors_per_conversation,
    "toots_per_conversation": toots_per_conversation,
    "conversations_per_instance": conversations_per_instance,
    "toots_per_author": toots_per_author,
    "cross_instance_toot_flows": federation_flows,
    "toxicity_per_instance": toxicity_per_instance,
}.items():
    table.to_csv(TABLES_DIR / f"{name}.csv", index=False)